In [1]:
"""Notebook Lens ML task: train a baseline model and save metrics."""

from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from samples.ml_tasks.common import (  # noqa: E402
    ARTIFACT_DIR,
    classification_metrics,
    generate_classification_rows,
    metrics_table,
    save_json,
    score_rows,
    split_rows,
    train_logistic,
)


ROW_COUNT = int(os.environ.get("ML_BASELINE_ROWS", "900"))
EPOCHS = int(os.environ.get("ML_BASELINE_EPOCHS", "140"))
LR = float(os.environ.get("ML_BASELINE_LR", "0.05"))
L2 = float(os.environ.get("ML_BASELINE_L2", "0.001"))

rows = generate_classification_rows(n=ROW_COUNT, seed=19)
train_rows, test_rows = split_rows(rows, train_frac=0.72, seed=3)
model = train_logistic(train_rows, lr=LR, epochs=EPOCHS, l2=L2)
scored_test = score_rows(test_rows, model)
metrics = classification_metrics(scored_test)

model_path = save_json(ARTIFACT_DIR / "baseline_model.json", model)
metrics_path = save_json(ARTIFACT_DIR / "baseline_metrics.json", metrics)

print(f"rows: {len(rows)}")
print(f"train_rows: {len(train_rows)}")
print(f"test_rows: {len(test_rows)}")
print(f"model_artifact: {model_path}")
print(f"metrics_artifact: {metrics_path}")
for point in model["history"]:
    print(f"epoch {point['epoch']:>3}: loss={point['loss']:.4f}")
print("test_metrics:")
for key, value in metrics.items():
    print(f"- {key}: {value:.4f}" if isinstance(value, float) else f"- {key}: {value}")

metrics_table("Baseline Test Metrics", metrics)
ML_TASK_STATUS = "ok"


rows: 900
train_rows: 648
test_rows: 252
model_artifact: artifacts/agent_rounds/20260615_round3/ml_artifact_env/baseline_model.json
metrics_artifact: artifacts/agent_rounds/20260615_round3/ml_artifact_env/baseline_metrics.json
epoch   1: loss=0.6931
epoch  23: loss=0.6256
epoch  46: loss=0.5823
epoch  69: loss=0.5548
epoch  92: loss=0.5365
epoch 115: loss=0.5237
epoch 138: loss=0.5146
epoch 140: loss=0.5139
test_metrics:
- tp: 108
- fp: 27
- tn: 89
- fn: 28
- precision: 0.8000
- recall: 0.7941
- accuracy: 0.7817
- f1: 0.7970
- brier: 0.1646


tp,108
fp,27
tn,89
fn,28
precision,0.8000
recall,0.7941
accuracy,0.7817
f1,0.7970
brier,0.1646


In [2]:
"""Notebook Lens ML task: run a small hyperparameter sweep."""

from __future__ import annotations

import itertools
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from samples.ml_tasks.common import (  # noqa: E402
    ARTIFACT_DIR,
    classification_metrics,
    display_html,
    generate_classification_rows,
    save_json,
    score_rows,
    split_rows,
    train_logistic,
)


ROW_COUNT = int(os.environ.get("ML_SWEEP_ROWS", "800"))
rows = generate_classification_rows(n=ROW_COUNT, seed=29)
train_rows, valid_rows = split_rows(rows, train_frac=0.7, seed=13)

learning_rates = [0.03, 0.06]
l2_values = [0.0, 0.001, 0.01]
epochs_values = [80, 140]
results = []

for lr, l2, epochs in itertools.product(learning_rates, l2_values, epochs_values):
    model = train_logistic(train_rows, lr=lr, l2=l2, epochs=epochs)
    metrics = classification_metrics(score_rows(valid_rows, model))
    result = {"lr": lr, "l2": l2, "epochs": epochs, **metrics}
    results.append(result)
    print(
        f"lr={lr:.3f} l2={l2:.3f} epochs={epochs} "
        f"f1={metrics['f1']:.4f} brier={metrics['brier']:.4f}"
    )

best = max(results, key=lambda row: (row["f1"], row["accuracy"], -row["brier"]))
artifact = save_json(ARTIFACT_DIR / "sweep_results.json", {"results": results, "best": best})

print(f"sweep_artifact: {artifact}")
print("best_config:")
for key in ["lr", "l2", "epochs", "f1", "accuracy", "precision", "recall", "brier"]:
    print(f"- {key}: {best[key]:.4f}" if isinstance(best[key], float) else f"- {key}: {best[key]}")

rows_html = "\n".join(
    "<tr>"
    f"<td>{row['lr']}</td><td>{row['l2']}</td><td>{row['epochs']}</td>"
    f"<td>{row['f1']:.3f}</td><td>{row['accuracy']:.3f}</td>"
    f"<td>{row['brier']:.3f}</td>"
    "</tr>"
    for row in sorted(results, key=lambda item: item["f1"], reverse=True)
)
display_html(
    "<h3>Hyperparameter Sweep</h3>"
    "<table>"
    "<tr><th>lr</th><th>l2</th><th>epochs</th><th>f1</th><th>accuracy</th><th>brier</th></tr>"
    f"{rows_html}"
    "</table>"
)
ML_TASK_STATUS = "ok"


lr=0.030 l2=0.000 epochs=80 f1=0.7586 brier=0.1882
lr=0.030 l2=0.000 epochs=140 f1=0.7586 brier=0.1737
lr=0.030 l2=0.001 epochs=80 f1=0.7586 brier=0.1883
lr=0.030 l2=0.001 epochs=140 f1=0.7586 brier=0.1738
lr=0.030 l2=0.010 epochs=80 f1=0.7586 brier=0.1887
lr=0.030 l2=0.010 epochs=140 f1=0.7586 brier=0.1743
lr=0.060 l2=0.000 epochs=80 f1=0.7586 brier=0.1708
lr=0.060 l2=0.000 epochs=140 f1=0.7619 brier=0.1625
lr=0.060 l2=0.001 epochs=80 f1=0.7586 brier=0.1709
lr=0.060 l2=0.001 epochs=140 f1=0.7619 brier=0.1625
lr=0.060 l2=0.010 epochs=80 f1=0.7619 brier=0.1715
lr=0.060 l2=0.010 epochs=140 f1=0.7619 brier=0.1632
sweep_artifact: artifacts/agent_rounds/20260615_round3/ml_artifact_env/sweep_results.json
best_config:
- lr: 0.0600
- l2: 0.0000
- epochs: 140
- f1: 0.7619
- accuracy: 0.7708
- precision: 0.7586
- recall: 0.7652
- brier: 0.1625


lr,l2,epochs,f1,accuracy,brier
0.06,0.0,140,0.762,0.771,0.162
0.06,0.001,140,0.762,0.771,0.163
0.06,0.01,80,0.762,0.771,0.172
0.06,0.01,140,0.762,0.771,0.163
0.03,0.0,80,0.759,0.767,0.188
0.03,0.0,140,0.759,0.767,0.174
0.03,0.001,80,0.759,0.767,0.188
0.03,0.001,140,0.759,0.767,0.174
0.03,0.01,80,0.759,0.767,0.189
0.03,0.01,140,0.759,0.767,0.174
